# Setup

In [5]:
import os
import sys
import time
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.85'
if os.getcwd().endswith("scripts/notebooks"):
    os.chdir("../../")
sys.path.append(os.getcwd())  
print(f"Directorio de trabajo actual: {os.getcwd()}")
print(f"Rutas de importación de Python: {sys.path[-1]}")

Directorio de trabajo actual: /home/alanh/Dev/owns/thesis
Rutas de importación de Python: /home/alanh/Dev/owns/thesis


In [6]:
import jax.numpy as jnp
import jax
import h5py
from tqdm.auto import tqdm
import csv
from tensorneat.common import State
import numpy as np

import importlib
import src.models.diffusion_network
import src.models.diffusion
import src.data
import src.loss
import src.config
import src.trainer
importlib.reload(src.models.diffusion_network)
importlib.reload(src.models.diffusion)
importlib.reload(src.config)
importlib.reload(src.models)
importlib.reload(src.data)
importlib.reload(src.loss)
importlib.reload(src.trainer)

from src.models.diffusion_network import CrystalDiffusionNetwork, BaseGenome, BaseMutation, BaseNEAT
from src.data import JAXBatchLoader
from src.loss import crystal_loss_fn
from src.trainer import Trainer

from src.config import *
KEY = jax.random.PRNGKey(SEED)

# Data

In [7]:
def load_crystal_dataset(h5_path):
    with h5py.File(h5_path, 'r') as hf:
        # 1. Cargar el Contexto Segmentado (Inputs inmutables)
        context_elements = hf['context_elements'][:]
        context_embeddings = hf['context_embeddings'][:]
        context_props = hf['context_props'][:]
        
        # 2. Cargar el Lienzo Cristalino Segmentado (Targets para Difusión)
        target_lattice = hf['target_lattice'][:]
        target_atoms = hf['target_atoms'][:] # Mantiene shape (N, MAX_ATOMS, 4)
        
        # 3. Cargar IDs de los materiales
        np_ids = hf['material_ids'][:]
        
    # Cálculo estadístico de consumo en RAM
    print(f"📦 Muestras Totales: {len(np_ids)}")
    print(f"📐 Shapes segmentados disponibles internamente:")
    print(f"   └─ Matrices de Átomos con Padding : {target_atoms.shape}")
    print(f"   └─ Parámetros del Lattice Global  : {target_lattice.shape}")
    print(f"   └─ Embeddings Cuánticos (CHGNet)  : {context_embeddings.shape}")
    
    # Devolvemos tanto el formato unificado plano clásico como las piezas difusivas segmentadas
    return {
        "ids": np_ids,
        "context_elements": context_elements,
        "context_embeddings": context_embeddings,
        "context_props": context_props,
        "target_lattice": target_lattice,
        "target_atoms": target_atoms
    }

dataset_dict = load_crystal_dataset(DATASET_PATH)

# Separamos cada componente manteniendo sus formas nativas
inputs_dict = {
    "elements": dataset_dict['context_elements'],       # (N, 18)
    "embeddings": dataset_dict['context_embeddings'],   # (N, 64)
    "props": dataset_dict['context_props']              # (N, 3)
}

targets_dict = {
    "lattice": dataset_dict['target_lattice'],  # (N, 6)
    "atoms": dataset_dict['target_atoms']       # (N, MAX_ATOMS, 4) Matrix!
}
material_ids = dataset_dict["ids"]                     # (N,)

# Cálculo de las dimensiones base para la red
CONTEXT_DIM = inputs_dict['elements'].shape[1] + inputs_dict['embeddings'].shape[1] + inputs_dict['props'].shape[1]
MAX_ATOMS = targets_dict['atoms'].shape[1] + targets_dict['lattice'].shape[1]

print(f"⚙️ CONTEXT_DIM Global (Suma de Inputs): {CONTEXT_DIM}")
print(f"⚙️ MAX_ATOMS con Padding en Matriz    : {MAX_ATOMS}")

train_loader = JAXBatchLoader(inputs_dict, targets_dict, material_ids, batch_size=BATCH_SIZE)

# Verificamos un lote de prueba para estar 100% seguros
batch_x, batch_y, batch_idx = next(train_loader)

print("✅ Pipeline de datos verificado con éxito:")
print(f"   Matrix de átomos en batch (Lienzo): {batch_y['atoms'].shape}") # Debería ser (8, 4, 4)
print(f"   Lattice en batch (Lienzo)         : {batch_y['lattice'].shape}") # Debería ser (8, 6)
print(f"   Embeddings en batch (Contexto)    : {batch_x['embeddings'].shape}") # Debería ser (8, 64)

print("INPUT DIM: ", INPUT_DIM)
print("OUTPUT DIM: ", OUTPUT_DIM)

📦 Muestras Totales: 84
📐 Shapes segmentados disponibles internamente:
   └─ Matrices de Átomos con Padding : (84, 4, 4)
   └─ Parámetros del Lattice Global  : (84, 6)
   └─ Embeddings Cuánticos (CHGNet)  : (84, 64)
⚙️ CONTEXT_DIM Global (Suma de Inputs): 85
⚙️ MAX_ATOMS con Padding en Matriz    : 10
✅ Pipeline de datos verificado con éxito:
   Matrix de átomos en batch (Lienzo): (8, 4, 4)
   Lattice en batch (Lienzo)         : (8, 6)
   Embeddings en batch (Contexto)    : (8, 64)
INPUT DIM:  85
OUTPUT DIM:  22


# Train

## Test Noise

In [5]:
import jax
import jax.numpy as jnp
import numpy as np
from tensorneat.common import State

# --- (Asegúrate de haber corrido las celdas anteriores que cargan los dicts y config) ---
from src.models.diffusion import CrystalDiffusion
from src.loss import noise_loss_fn, crystal_loss_fn
from src.models.diffusion_network import CrystalDiffusionNetwork, BaseGenome, BaseMutation, BaseNEAT
from src.config import *

# 1. INICIALIZAR EL ORÁCULO (DIFUSIÓN) Y EL CEREBRO
neat_config = BaseNEAT(
    pop_size=POPSIZE,
    species_size=SPECIES_SIZE,
    survival_threshold=SURVIVAL_THRESHOLD
)

genome_config = BaseGenome(
    max_nodes=MAX_NODES,
    max_conns=MAX_CONNS
)

mutation_config = BaseMutation(
    conn_add_prob=CONN_ADD_PROB,
    conn_delete_prob=CONN_DELETE_PROB,
    node_add_prob=NODE_ADD_PROB,
    node_delete_prob=NODE_DELETE_PROB
)
model = CrystalDiffusionNetwork(
    elements_dim=MAX_ELEMENTS * ELEM_FEATURES,  # 18
    embeddings_dim=CRYSTAL_EMBED,               # 64
    props_dim=CRYSTAL_PROPS,                    # 3
    lattice_dim=LATTICE_PARAMS,                 # 6
    atom_dim=4,                                 # Z, x, y, z
    neat_config=neat_config,
    genome_config=genome_config,
    mutation_config=mutation_config
)
diffusion = CrystalDiffusion()

# 2. EXTRAER UN CRISTAL DEL DATASET (Batch = 1)
elements = inputs_dict['elements'][0][np.newaxis, :]
embeddings = inputs_dict['embeddings'][0][np.newaxis, :]
props = inputs_dict['props'][0][np.newaxis, :]
target_lattice = targets_dict['lattice'][0][np.newaxis, :]
target_atoms = targets_dict['atoms'][0][np.newaxis, :]

mask = (target_atoms[:, :, 0] > 0).astype(jnp.float32)

# 3. INYECTAR EL RUIDO
key = jax.random.PRNGKey(SEED)
key_lat, key_atom, key_err, key_neat = jax.random.split(key, 4)
t = 0.5 

noisy_lattice, true_noise_lattice = diffusion.add_noise(target_lattice, key_lat, t)
noisy_atoms, true_noise_atoms = diffusion.add_noise(target_atoms, key_atom, t)

# ==============================================================================
# PRUEBA 1: COMPROBANDO LA FUNCIÓN NOISE_LOSS_FN
# ==============================================================================
print("====== 🧪 TEST 1: NOISE LOSS (Lamarckismo) 🧪 ======")

# A) Perfecta
loss_perfecta = noise_loss_fn(true_noise_lattice, true_noise_atoms, true_noise_lattice, true_noise_atoms, mask)
print(f"✅ Escenario 1 (Perfecta)           -> Loss: {loss_perfecta:.4f}")

# B) 25% Error
noise_error_25_lat = true_noise_lattice * 1.25
noise_error_25_atoms = true_noise_atoms * 1.25
loss_error_25 = noise_loss_fn(noise_error_25_lat, noise_error_25_atoms, true_noise_lattice, true_noise_atoms, mask)
print(f"⚠️ Escenario 2 (25% Error)          -> Loss: {loss_error_25:.4f}")

# C) Random
random_noise_lat = jax.random.normal(key_err, shape=true_noise_lattice.shape)
random_noise_atoms = jax.random.normal(key_err, shape=true_noise_atoms.shape)
loss_error_random = noise_loss_fn(random_noise_lat, random_noise_atoms, true_noise_lattice, true_noise_atoms, mask)
print(f"🛑 Escenario 3 (Aleatoria)          -> Loss: {loss_error_random:.4f}")

# D) Individuo NEAT (Seleccionamos el primero de la población)
# Usamos el estado inicializado del modelo
state = State(randkey=key_neat)
state = model.neat.setup(state)
pop_nodes, pop_conns = model.neat.ask(state)

# Extraemos solo el primero de los 32 individuos
node_ind0 = pop_nodes[0]
conn_ind0 = pop_conns[0]

pred_noise_lattice, pred_noise_atoms = model.forward_crystal(
    state, (node_ind0, conn_ind0), jnp.array([t]), elements[0], embeddings[0], props[0], noisy_lattice[0], noisy_atoms[0]
)

# Ajustamos dimensiones para noise_loss_fn que espera (Batch, ...)
loss_neat_init = noise_loss_fn(
    pred_noise_lattice[np.newaxis, :], pred_noise_atoms[np.newaxis, :], 
    true_noise_lattice, true_noise_atoms, mask
)
print(f"🤖 Escenario 4 (NEAT Inicial)       -> Loss: {loss_neat_init:.4f}")

# ==============================================================================
# PRUEBA 2: COMPROBANDO LA FUNCIÓN CRYSTAL_LOSS_FN
# ==============================================================================
print("\n====== 🧪 TEST 2: CRYSTAL LOSS (Darwinismo) 🧪 ======")

flat_target_atoms = target_atoms.reshape(1, -1)
targets_flat = jnp.concatenate([target_lattice, flat_target_atoms], axis=-1)

# A) Perfecto
loss_c_perfecta = crystal_loss_fn(targets_flat, targets_flat, separate_results=True)
print(f"✅ Escenario 1 (Cristal Perfecto)   -> Loss: {loss_c_perfecta[0]:.4f}")

# B) 25% Error
denoised_lat_error25 = diffusion.predict_crystal(noisy_lattice, noise_error_25_lat, t)
denoised_atoms_error25 = diffusion.predict_crystal(noisy_atoms, noise_error_25_atoms, t)
preds_flat_err25 = jnp.concatenate([denoised_lat_error25, denoised_atoms_error25.reshape(1, -1)], axis=-1)
loss_c_err25 = crystal_loss_fn(preds_flat_err25, targets_flat, separate_results=True)
print(f"⚠️ Escenario 2 (25% Error)          -> Loss: {loss_c_err25[0]:.4f}")

# C) NEAT
denoised_lat_neat = diffusion.predict_crystal(noisy_lattice, pred_noise_lattice[np.newaxis, :], t)
denoised_atoms_neat = diffusion.predict_crystal(noisy_atoms, pred_noise_atoms[np.newaxis, :], t)
preds_flat_neat = jnp.concatenate([denoised_lat_neat, denoised_atoms_neat.reshape(1, -1)], axis=-1)
loss_c_neat = crystal_loss_fn(preds_flat_neat, targets_flat, separate_results=True)
print(f"🤖 Escenario 3 (Reconstrucción NEAT)-> Loss: {loss_c_neat[0]:.4f}")

====== 🧪 TEST 1: NOISE LOSS (Lamarckismo) 🧪 ======
✅ Escenario 1 (Perfecta)           -> Loss: 0.0000
⚠️ Escenario 2 (25% Error)          -> Loss: 0.2187
🛑 Escenario 3 (Aleatoria)          -> Loss: 4.3228


TypeError: CrystalDiffusionNetwork.forward_crystal() missing 1 required positional argument: 'noisy_atoms'

## Test

: 

: 

: 

: 

## Train Setup

In [8]:
# ======================================================================
# CELDA 4: INICIALIZACIÓN Y BUCLE DE ENTRENAMIENTO
# ======================================================================

# 1. Configurar las estructuras de TensorNEAT usando src.config
neat_config = BaseNEAT(
    pop_size=POPSIZE,
    species_size=SPECIES_SIZE,
    survival_threshold=SURVIVAL_THRESHOLD
)

genome_config = BaseGenome(
    max_nodes=MAX_NODES,
    max_conns=MAX_CONNS
)

mutation_config = BaseMutation(
    conn_add_prob=CONN_ADD_PROB,
    conn_delete_prob=CONN_DELETE_PROB,
    node_add_prob=NODE_ADD_PROB,
    node_delete_prob=NODE_DELETE_PROB
)

# 2. Inicializar el Cerebro (Point-Cloud Network Condicionada)
model = CrystalDiffusionNetwork(
    elements_dim=MAX_ELEMENTS * ELEM_FEATURES,  # 18
    embeddings_dim=CRYSTAL_EMBED,               # 64
    props_dim=CRYSTAL_PROPS,                    # 3
    lattice_dim=LATTICE_PARAMS,                 # 6
    atom_dim=4,                                 # Z, x, y, z
    neat_config=neat_config,
    genome_config=genome_config,
    mutation_config=mutation_config
)

# 3. Inicializar el Entrenador (Lamarckismo + Darwinismo)
# Creamos una subcarpeta 'logs' dentro de tu carpeta principal de datos
LOGS_DIR = "runs/v2/"

trainer = Trainer(
    model=model,
    logs_path=LOGS_DIR,
    n_generations=N_GENERATIONS,
    grad_steps_per_gen=GRAD_STEPS_PER_GEN,
    seed=SEED,
    lr_min=LR_MIN,
    lr_max=LR_MAX
)


# 4. ¡Arrancar la Evolución Difusiva!
print(f"🚀 Iniciando entrenamiento difusivo Lamarckiano...")
print(f"   └─ Generaciones: {N_GENERATIONS}")
print(f"   └─ Grad Steps por Gen: {GRAD_STEPS_PER_GEN}")
print(f"   └─ Población NEAT: {POPSIZE}")
print(f"   └─ Batch Size: {BATCH_SIZE}\n")

# NOTA MENTAL: La primera generación tardará un poco más porque JAX
# debe compilar el grafo computacional (JIT) para tu Mac M4.
# A partir de la generación 2, volará.
trainer.fit(train_loader)

🚀 Iniciando entrenamiento difusivo Lamarckiano...
   └─ Generaciones: 50
   └─ Grad Steps por Gen: 10
   └─ Población NEAT: 64
   └─ Batch Size: 8



🧬 Escultores Lamarckianos:   0%|          | 0/50 [00:00<?, ?it/s]


📊 ¡Terminado! 5 Gráficos generados exitosamente en:
📁 runs/v2/training_logs_plots


In [ ]:
from src.inference import CrystalGenerator

# 1. Instanciamos el generador cargando el archivo .npz
generator = CrystalGenerator(
    model_network=model, # Tu modelo ya definido
    weights_path=os.path.join(LOGS_DIR, "best_genome.npz")
)

# 2. Tomamos el contexto deseado (ej. inventamos un material)
# Aquí usamos uno del dataset como prueba
test_elements = inputs_dict['elements'][0]
test_embeddings = inputs_dict['embeddings'][0]
test_props = inputs_dict['props'][0]

# 3. ¡Hágase la luz! (Generamos el cristal en 50 pasos)
gen_lattice, gen_atoms = generator.generate(
    elements=test_elements, 
    embeddings=test_embeddings, 
    props=test_props,
    num_steps=50  # Bucle de Denoising (DDIM)
)

print("\n💎 CRISTAL GENERADO 💎")
print("Lattice:", gen_lattice)
print("Átomos (Z, x, y, z):")
print(gen_atoms)

: 

: 

: 

: 